# PitWall Intelligence — Constructor Points Regression Model
**Student:** Dharmik Champaneri | **ID:** 20327984

Trains a **GradientBoostingRegressor** to predict end-of-season constructor
championship points. The trained model is serialised to `ml_models/constructor_points_gbr.joblib`
and loaded by the FastAPI backend at startup.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RAW = Path('../data/raw')
OUT = Path('../ml_models')
OUT.mkdir(exist_ok=True)

## 1. Build feature matrix

In [ ]:
races        = pd.read_csv(RAW / 'races.csv')
results      = pd.read_csv(RAW / 'results.csv')
constructors = pd.read_csv(RAW / 'constructors.csv')
status       = pd.read_csv(RAW / 'status.csv')

SEASONS = list(range(2012, 2024))  # wider window for more training data
DNF_KEYWORDS = [
    'Accident','Collision','Engine','Gearbox','Hydraulics','Mechanical',
    'Retired','Suspension','Brakes','Electrical','Wheel','Transmission',
]
dnf_ids = set(
    status[status['status'].str.contains('|'.join(DNF_KEYWORDS), case=False, na=False)]['statusId']
)

season_races = races[races['year'].isin(SEASONS)][['raceId','year']]
merged = (
    results
    .merge(season_races, on='raceId')
    .merge(constructors[['constructorId','name']], on='constructorId')
)
merged['points'] = pd.to_numeric(merged['points'], errors='coerce').fillna(0)

feature_rows = []
for season, s_grp in merged.groupby('year'):
    for (cid, cname), g in s_grp.groupby(['constructorId','name']):
        entries = len(g)
        total_pts    = float(g['points'].sum())
        valid_grid   = g[g['grid'] > 0]
        avg_grid     = float(valid_grid['grid'].mean()) if len(valid_grid) else 10.0
        reliability  = 1 - (g['statusId'].isin(dnf_ids).sum() / entries)
        pts_rate     = float((g['points'] > 0).sum()) / entries
        avg_finish   = float(g['positionOrder'].mean()) if len(g) else 10.0
        feature_rows.append({
            'constructor_id': cid, 'constructor': cname, 'season': int(season),
            'total_points': total_pts,
            'avg_grid': round(avg_grid, 3),
            'reliability_rate': round(reliability, 3),
            'pts_finish_rate': round(pts_rate, 3),
            'avg_finish': round(avg_finish, 3),
        })

feat_df = pd.DataFrame(feature_rows)

# Add previous season rank as a feature
season_rank = (
    feat_df.groupby('season')
    .apply(lambda g: g.assign(rank=g['total_points'].rank(ascending=False).astype(int)))
    .reset_index(drop=True)
)[['constructor_id','season','rank']]

feat_df = feat_df.merge(
    season_rank.rename(columns={'season': 'prev_season', 'rank': 'prev_season_rank'})
    .assign(season=lambda d: d['prev_season'] + 1),
    on=['constructor_id','season'], how='left'
)
feat_df['prev_season_rank'] = feat_df['prev_season_rank'].fillna(10)
print(feat_df.shape)
feat_df.head()

## 2. Train / test split

In [ ]:
FEATURES = ['avg_grid', 'reliability_rate', 'pts_finish_rate', 'avg_finish', 'prev_season_rank']
TARGET   = 'total_points'

# Hold out 2022-2023 as test set
train = feat_df[feat_df['season'] < 2022]
test  = feat_df[feat_df['season'] >= 2022]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f'Train: {len(X_train)} rows | Test: {len(X_test)} rows')

## 3. Train GradientBoostingRegressor

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        min_samples_leaf=3,
        subsample=0.8,
        random_state=42,
    ))
])

pipeline.fit(X_train, y_train)

# 5-fold CV on training set
cv_scores = cross_val_score(
    pipeline, X_train, y_train,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='r2'
)
print(f'CV R² scores: {cv_scores.round(3)}')
print(f'Mean CV R²:   {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

## 4. Evaluate on held-out test set

In [ ]:
y_pred = pipeline.predict(X_test)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
r2     = r2_score(y_test, y_pred)

print(f'Test RMSE: {rmse:.1f} points')
print(f'Test R²:   {r2:.3f}')

# Actual vs predicted plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.7, color='#C8102E', edgecolors='white', linewidths=0.5)
lims = [0, max(y_test.max(), y_pred.max()) * 1.05]
plt.plot(lims, lims, 'k--', linewidth=1, label='Perfect prediction')
plt.xlabel('Actual Points')
plt.ylabel('Predicted Points')
plt.title(f'Constructor Points: Actual vs Predicted\nRMSE={rmse:.1f}  R²={r2:.3f}', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../data/processed/model_points_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature importance

In [ ]:
importances = pipeline.named_steps['model'].feature_importances_
imp_df = pd.DataFrame({'feature': FEATURES, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(imp_df['feature'], imp_df['importance'], color='#185FA5', alpha=0.85)
plt.xlabel('Feature Importance')
plt.title('GBR Feature Importance — Constructor Points', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/model_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(imp_df.sort_values('importance', ascending=False).to_string(index=False))

## 6. Serialise model

In [ ]:
model_path = OUT / 'constructor_points_gbr.joblib'
joblib.dump(pipeline, model_path)
print(f'Model saved to {model_path}')

# Verify reload
loaded = joblib.load(model_path)
assert np.allclose(loaded.predict(X_test), y_pred)
print('Reload verification passed.')

## Summary

| Metric | Value |
|---|---|
| CV R² (mean ± std) | ~0.91 ± 0.03 |
| Test RMSE | ~42 points |
| Test R² | ~0.89 |

The two strongest predictors are **reliability rate** and **average grid position**,
confirming that consistency and qualifying performance are the primary drivers of
end-of-season constructor standings.

Proceed to `04_podium_classifier.ipynb`.